In [1]:
import os
import numpy as np
from tqdm import tqdm
from glob import glob
from concurrent.futures import ThreadPoolExecutor, as_completed
from utils.data_processing import remove_padding

input_dir = 'your_directory/bucket/dnstream/input'
output_dir = 'your_directory/loads/weather'
os.makedirs(output_dir, exist_ok=True)

pmask = np.load('your_directory/bucket/dnstream/stats/pmask.npz', allow_pickle=True)['pmask']


county_mask_dict = np.load('your_directory/loads/county_masks_conus.npz')
county_codes = list(county_mask_dict.keys())

county_indices = {}
for code in county_codes:
    mask = county_mask_dict[code].astype(bool)
    idx = np.nonzero(mask)  # tuple of (rows, cols)
    county_indices[code] = idx

input_files = sorted(glob(os.path.join(input_dir, 'input_2018*.npz')))
T = len(input_files)

results = {code: [] for code in county_codes}

In [ ]:
def process_one_file(filepath):
    try:
        data = np.load(filepath)['input']  # shape (C, H, W)
        data = remove_padding(data, pmask)
        C = data.shape[0]
        file_result = {}

        for code in county_codes:
            r_idx, c_idx = county_indices[code]
            county_mean = []
            for c in range(C):
                values = data[c, r_idx, c_idx]
                values = values[~np.isnan(values)]
                county_mean.append(values.mean() if len(values) > 0 else np.nan)
            file_result[code] = county_mean

        return filepath, file_result
    except Exception as e:
        print(f"[Warning] Failed to process {filepath}: {e}")
        return filepath, None

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(process_one_file, f) for f in input_files]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing files"):
        filepath, file_result = future.result()
        if file_result is None:
            continue
        for code in county_codes:
            results[code].append(file_result[code])


for code in tqdm(county_codes, desc="Saving"):
    arr = np.array(results[code])  # shape [T, C]
    np.savez_compressed(os.path.join(output_dir, f'weather_{code}.npz'), data=arr)

In [ ]:
input_dir = 'your_directory/bucket/dnstream/input'
output_dir = 'your_directory/loads/weather'
pmask = np.load('your_directory/bucket/dnstream/stats/pmask.npz', allow_pickle=True)['pmask']

county_mask_dict = np.load('your_directory/loads/county_masks_conus.npz')
county_codes = list(county_mask_dict.keys())
county_indices = {code: np.nonzero(county_mask_dict[code].astype(bool)) for code in county_codes}

input_files = sorted(glob(os.path.join(input_dir, 'input_20190101*.npz')))
print(f"Found {len(input_files)} input files for 2019-01-01")

new_results = {code: [] for code in county_codes}

for filepath in tqdm(input_files, desc="Processing 2019-01-01"):
    data = np.load(filepath)['input']
    data = remove_padding(data, pmask)
    C = data.shape[0]

    for code in county_codes:
        r_idx, c_idx = county_indices[code]
        county_mean = []
        for c in range(C):
            values = data[c, r_idx, c_idx]
            values = values[~np.isnan(values)]
            county_mean.append(values.mean() if len(values) > 0 else np.nan)
        new_results[code].append(county_mean)

for code in tqdm(county_codes, desc="Merging and Saving"):
    old_path = os.path.join(output_dir, f'weather_{code}.npz')
    if not os.path.exists(old_path):
        print(f"[Warning] Missing file for county {code}, skipping.")
        continue

    old_data = np.load(old_path)['data']  # shape [T_old, C]
    new_data = np.array(new_results[code])  # shape [24, C]
    merged = np.concatenate([old_data, new_data], axis=0)
    np.savez_compressed(old_path, data=merged)
